# 탐 그룹 규칙 값 다시 고르기 — GPU 서버 실행 (설계 CHART-003-tom-rule-retune)

11/07 로컬 RTX 4070에서 한 실행(`docs/chart-rules/tom-rule-retune-2026-11/summary.md`)을 서버에서 다시 할 때 쓴다.

- 특징: E-GMD test 어쿠스틱 1,170개, ENST 드럼 단독 64곡, ENST 반주 1.0 원곡 → BS Roformer SW 분리 드럼 64곡. 없는 것만 GPU 6번으로 만든다
- 채점: E-GMD에 후보 92개, ENST 두 묶음에 후보 3개(current·current_order·E-GMD에서 고른 후보). 곡 하나씩 읽어 rule-eval과 같은 계산으로 채점
- 고르기·ENST 차이 계산은 로컬 `scripts/tom_rule_pick.py`로 한다

**준비물.** 이 파일만 JupyterHub `ssaplash/notebooks/`에 올린다. 서버 `~/ssaplash/S15P31A307`은 운영 워커 코드라 덮어쓰지 않고, 그 `.venv`와 `ssaplash-bench rule-features`를 읽기·실행만 한다. 채점 스크립트와 후보 yaml은 2번 셀이 `~/ssaplash/tom_rule/`에 새로 쓴다. 특징 폴더는 `~/ssaplash/tom_rule/features/`에 따로 만든다.

**순서.** 1 설정 → 2 스크립트·후보 쓰기 → 3 데이터·특징 점검 → 4 특징 만들기(백그라운드, 한 번만) → 5 진행 확인(여러 번) → 6 채점 시작(백그라운드, 한 번만) → 7 결과 확인·묶기.

**끊기면.** 4·6번은 nohup이라 브라우저를 닫아도 끝까지 돈다. 커널이 다시 시작됐으면 1번만 다시 치고 5번 또는 7번부터 이어 간다.

In [ ]:
# 1. 설정. 지정받은 GPU는 6번
import getpass
import os

GPU_ID = "6"
os.environ["CUDA_VISIBLE_DEVICES"] = GPU_ID
os.environ["PYTHONUTF8"] = "1"
os.environ["PYTHONUNBUFFERED"] = "1"
os.environ["OMP_NUM_THREADS"] = "1"
os.environ["OPENBLAS_NUM_THREADS"] = "1"
os.environ["MKL_NUM_THREADS"] = "1"
HOME = os.path.expanduser("~")
WORK = f"{HOME}/ssaplash"
AI = f"{WORK}/S15P31A307/AI"            # 운영 워커 코드. 읽기·실행만
VPY = f"{AI}/.venv/bin/python"
BENCH = f"{AI}/.venv/bin/ssaplash-bench"
TOOL = f"{WORK}/tom_rule"
FEAT = f"{TOOL}/features"               # 드럼 단독 특징
FEAT_SEP = f"{TOOL}/features_sep"       # 분리 입력 특징
OUT = f"{WORK}/results/tom-rule-retune"
LOG_FEAT = f"{TOOL}/features.log"
LOG_EVAL = f"{TOOL}/eval.log"
ME = getpass.getuser()
DS = "configs/bench/datasets"
SEP_MODEL = "configs/bench/models/msst/bs_roformer_sw.yaml"
WORKERS = 32
for d in (TOOL, OUT):
    os.makedirs(d, exist_ok=True)
%cd $AI
print("CPU 코어", os.cpu_count(), "· 채점 작업", WORKERS)
!git -C $AI/.. log --oneline -1
!ls -d data/e-gmd/e-gmd-v1.0.0 data/enst/ENST-drums-public data/models/bs_roformer_sw data/models/mdx23c_drumsep
!nvidia-smi -i $GPU_ID --query-gpu=index,name,memory.used,memory.total --format=csv,noheader
# 확인: 커밋 줄 하나, 경로 4개, GPU 6번 줄. ENST 경로가 없으면 data/enst 링크가 없는 것이라 그 줄을 적어 두고 멈춤

In [ ]:
# 2. 채점 스크립트·후보 yaml 쓰기 (여러 번 쳐도 됨. 같은 내용으로 덮어씀)
FILES = {
    f"{TOOL}/tom_rule_eval_stream.py": r'''
"""탐 그룹 규칙 후보를 곡 하나씩 읽어 채점 (설계 CHART-003-tom-rule-retune v0.3).

rule-eval(`ssaplash-bench rule-eval`)은 데이터셋 특징을 전부 메모리에 올린 뒤 후보를 돌린다. E-GMD test 1,170개는
분리 트랙 6개가 22.7시간 분량이라 로컬 메모리(64GB)에 다 안 올라감. 이 스크립트는 곡마다 특징을 읽고 후보 전부를
같은 rule_eval.eval_song으로 채점한 뒤 버림. 곡별 개수는 rule-eval by_song.csv와 같은 계산이다.

- 분리 입력 특징(`--separator`로 만든 것, stem_input 있음)은 그 분리 드럼을 거르기 입력으로 씀. 없으면 데이터셋의
  정답 드럼 트랙을 씀(rule-eval과 같음)
- 출력 csv 한 줄 = (후보, 곡). 키트 이름(E-GMD·STAR)과 정답 탐 종류 수를 같이 적음

  uv run python scripts/tom_rule_eval_stream.py --dataset configs/bench/datasets/egmd_test_acoustic.yaml \
      --features data/rule_features --candidates docs/.../candidates_part1.yaml --two-tom-upper MID_TOM \
      --workers 16 --out data/results/tom-rule-retune/egmd_test.csv
"""

from __future__ import annotations

import argparse
import csv
import os
import time
from multiprocessing import Pool
from pathlib import Path

import numpy as np

TOM_PADS = ("SMALL_TOM", "MID_TOM", "FLOOR_TOM")
# rule_features.INPUT_STEM과 같은 값. 서버 운영 코드(6023e0e)에는 이 상수가 없어 여기서 정함
INPUT_STEM = "input"
_STATE: dict = {}


def _init(
    dataset_yaml: str, features_root: str, candidates: list, two_tom_upper: str | None
) -> None:
    from ssaplash_ai.bench.datasets import DatasetConfig, load_dataset

    _STATE.update(
        dataset=load_dataset(DatasetConfig.from_yaml(dataset_yaml)),
        features_root=Path(features_root),
        candidates=candidates,
        two_tom_upper=two_tom_upper,
    )


def song_rows(song_id: str) -> list[dict]:
    """곡 하나 × 후보 전부 → csv 줄 목록"""
    from ssaplash_ai.bench.analysis.rule_eval import eval_song, rename_two_tom_upper
    from ssaplash_ai.bench.analysis.rule_features import (
        feature_path,
        input_as_feature,
        load_features,
    )
    from ssaplash_ai.bench.datasets.pads import SCORED_PADS

    ds = _STATE["dataset"]
    path = feature_path(_STATE["features_root"], ds.name, song_id)
    if not path.is_file():
        raise FileNotFoundError(f"특징 파일 없음: {path}")
    hits_true, _ = ds.hits_of(song_id)
    if _STATE["two_tom_upper"]:
        hits_true = rename_two_tom_upper(hits_true, _STATE["two_tom_upper"])
    feats = load_features(path)
    if INPUT_STEM in feats.stems:
        feats.input_audio = feats.stems.pop(INPUT_STEM).astype(np.float16)
    else:
        feats.input_audio = input_as_feature(ds.load(song_id).drums_true)
    kit = ds.kit_of(song_id) if hasattr(ds, "kit_of") else ""
    n_kinds = len({str(p) for p in hits_true["pad"] if str(p) in TOM_PADS})
    rows = []
    for cand in _STATE["candidates"]:
        ev = eval_song(song_id, feats, hits_true, cand["rules"])
        row = {
            "candidate": cand["name"],
            "dataset": ds.name,
            "songId": song_id,
            "kit": kit,
            "tomTrueKinds": n_kinds,
            "tomMatched": ev.tom.matched,
            "tomCorrect": ev.tom.correct,
            "tomFalseKept": ev.tom.false_kept,
            "tomMissed": ev.tom.missed,
            "tomClusters": ev.tom_clusters,
            "tomClusterHz": ev.tom_cluster_hz,
        }
        for p in SCORED_PADS:
            c = ev.pads.get(p)
            row[f"tp{p}"], row[f"fp{p}"], row[f"fn{p}"] = (c.tp, c.fp, c.fn) if c else (0, 0, 0)
        rows.append(row)
    return rows


def load_candidates(paths: list[Path]) -> list[dict]:
    from ssaplash_ai.bench.analysis.rule_eval import load_candidates as _load

    return [{"name": c.name, "rules": c.rules} for c in _load(paths)]


def main() -> int:
    ap = argparse.ArgumentParser(description=__doc__.splitlines()[0])
    ap.add_argument("--dataset", required=True, help="데이터셋 yaml (정답용)")
    ap.add_argument("--features", default="data/rule_features", help="rule-features 출력 폴더")
    ap.add_argument("--candidates", nargs="+", type=Path, required=True, help="후보 yaml")
    ap.add_argument("--two-tom-upper", choices=["MID_TOM", "SMALL_TOM"], default=None)
    ap.add_argument("--workers", type=int, default=max(1, (os.cpu_count() or 2) // 2))
    ap.add_argument("--limit", type=int, default=None, help="앞에서 이만큼 곡만 (확인용)")
    ap.add_argument("--out", type=Path, required=True, help="출력 csv")
    args = ap.parse_args()
    from ssaplash_ai.bench.datasets import DatasetConfig, load_dataset

    cands = load_candidates(args.candidates)
    songs = load_dataset(DatasetConfig.from_yaml(args.dataset)).song_ids()
    if args.limit:
        songs = songs[: args.limit]
    print(f"[tom-rule] 곡 {len(songs)}개 × 후보 {len(cands)}개, 작업 {args.workers}", flush=True)
    args.out.parent.mkdir(parents=True, exist_ok=True)
    t0 = time.time()
    init = (args.dataset, args.features, cands, args.two_tom_upper)
    with open(args.out, "w", newline="", encoding="utf-8") as f:
        writer = None

        def write(i: int, rows: list[dict]) -> None:
            nonlocal writer
            if writer is None:
                writer = csv.DictWriter(f, fieldnames=list(rows[0].keys()))
                writer.writeheader()
            writer.writerows(rows)
            if i % 50 == 0 or i == len(songs):
                print(f"[tom-rule] {i}/{len(songs)}곡 ({time.time() - t0:.0f}초)", flush=True)

        if args.workers <= 1:
            # 작업 1개면 프로세스를 안 띄움 (테스트처럼 파일 경로로 불러온 모듈도 돌게)
            _init(*init)
            for i, song in enumerate(songs, 1):
                write(i, song_rows(song))
        else:
            with Pool(args.workers, initializer=_init, initargs=init) as pool:
                for i, rows in enumerate(pool.imap_unordered(song_rows, songs), 1):
                    write(i, rows)
    print(f"[tom-rule] 저장 {args.out}", flush=True)
    return 0


if __name__ == "__main__":
    raise SystemExit(main())
''',
    f"{TOOL}/candidates_part1.yaml": r'''
candidates:
- name: current
  rules:
    window_ms: 120
    cymbal_window_ms: 80
    tom_fmin: 50
    tom_fmax: 400
    tom_bands:
    - 113
    - 142
    tom_gate_ratio: 0.1
    tom_peak_mode: raw
    tom_mode: relative
    tom_relative_method: peaks
    tom_relative_naming: hybrid
    tom_relative_min_gap: 3.0
    cymbal_crash_weight: 0.7
    stems:
      toms: toms
      crash: crash
      ride: ride
- name: current_order
  rules:
    window_ms: 120
    cymbal_window_ms: 80
    tom_fmin: 50
    tom_fmax: 400
    tom_bands:
    - 113
    - 142
    tom_gate_ratio: 0.1
    tom_peak_mode: raw
    tom_mode: relative
    tom_relative_method: peaks
    tom_relative_naming: order
    tom_relative_min_gap: 3.0
    cymbal_crash_weight: 0.7
    stems:
      toms: toms
      crash: crash
      ride: ride
- name: order_gap15_sm025_fr00_peak
  rules:
    window_ms: 120
    cymbal_window_ms: 80
    tom_fmin: 50
    tom_fmax: 400
    tom_bands:
    - 113
    - 142
    tom_gate_ratio: 0.1
    tom_peak_mode: raw
    tom_mode: relative
    tom_relative_method: peaks
    tom_relative_naming: order
    tom_relative_min_gap: 1.5
    cymbal_crash_weight: 0.7
    stems:
      toms: toms
      crash: crash
      ride: ride
    tom_relative_smooth: 0.25
    tom_relative_min_frac: 0.0
    tom_f0_method: peak
- name: order_gap15_sm025_fr00_median
  rules:
    window_ms: 120
    cymbal_window_ms: 80
    tom_fmin: 50
    tom_fmax: 400
    tom_bands:
    - 113
    - 142
    tom_gate_ratio: 0.1
    tom_peak_mode: raw
    tom_mode: relative
    tom_relative_method: peaks
    tom_relative_naming: order
    tom_relative_min_gap: 1.5
    cymbal_crash_weight: 0.7
    stems:
      toms: toms
      crash: crash
      ride: ride
    tom_relative_smooth: 0.25
    tom_relative_min_frac: 0.0
    tom_f0_method: median
- name: order_gap15_sm025_fr05_peak
  rules:
    window_ms: 120
    cymbal_window_ms: 80
    tom_fmin: 50
    tom_fmax: 400
    tom_bands:
    - 113
    - 142
    tom_gate_ratio: 0.1
    tom_peak_mode: raw
    tom_mode: relative
    tom_relative_method: peaks
    tom_relative_naming: order
    tom_relative_min_gap: 1.5
    cymbal_crash_weight: 0.7
    stems:
      toms: toms
      crash: crash
      ride: ride
    tom_relative_smooth: 0.25
    tom_relative_min_frac: 0.05
    tom_f0_method: peak
- name: order_gap15_sm025_fr05_median
  rules:
    window_ms: 120
    cymbal_window_ms: 80
    tom_fmin: 50
    tom_fmax: 400
    tom_bands:
    - 113
    - 142
    tom_gate_ratio: 0.1
    tom_peak_mode: raw
    tom_mode: relative
    tom_relative_method: peaks
    tom_relative_naming: order
    tom_relative_min_gap: 1.5
    cymbal_crash_weight: 0.7
    stems:
      toms: toms
      crash: crash
      ride: ride
    tom_relative_smooth: 0.25
    tom_relative_min_frac: 0.05
    tom_f0_method: median
- name: order_gap15_sm025_fr10_peak
  rules:
    window_ms: 120
    cymbal_window_ms: 80
    tom_fmin: 50
    tom_fmax: 400
    tom_bands:
    - 113
    - 142
    tom_gate_ratio: 0.1
    tom_peak_mode: raw
    tom_mode: relative
    tom_relative_method: peaks
    tom_relative_naming: order
    tom_relative_min_gap: 1.5
    cymbal_crash_weight: 0.7
    stems:
      toms: toms
      crash: crash
      ride: ride
    tom_relative_smooth: 0.25
    tom_relative_min_frac: 0.1
    tom_f0_method: peak
- name: order_gap15_sm025_fr10_median
  rules:
    window_ms: 120
    cymbal_window_ms: 80
    tom_fmin: 50
    tom_fmax: 400
    tom_bands:
    - 113
    - 142
    tom_gate_ratio: 0.1
    tom_peak_mode: raw
    tom_mode: relative
    tom_relative_method: peaks
    tom_relative_naming: order
    tom_relative_min_gap: 1.5
    cymbal_crash_weight: 0.7
    stems:
      toms: toms
      crash: crash
      ride: ride
    tom_relative_smooth: 0.25
    tom_relative_min_frac: 0.1
    tom_f0_method: median
- name: order_gap15_sm050_fr00_peak
  rules:
    window_ms: 120
    cymbal_window_ms: 80
    tom_fmin: 50
    tom_fmax: 400
    tom_bands:
    - 113
    - 142
    tom_gate_ratio: 0.1
    tom_peak_mode: raw
    tom_mode: relative
    tom_relative_method: peaks
    tom_relative_naming: order
    tom_relative_min_gap: 1.5
    cymbal_crash_weight: 0.7
    stems:
      toms: toms
      crash: crash
      ride: ride
    tom_relative_smooth: 0.5
    tom_relative_min_frac: 0.0
    tom_f0_method: peak
- name: order_gap15_sm050_fr00_median
  rules:
    window_ms: 120
    cymbal_window_ms: 80
    tom_fmin: 50
    tom_fmax: 400
    tom_bands:
    - 113
    - 142
    tom_gate_ratio: 0.1
    tom_peak_mode: raw
    tom_mode: relative
    tom_relative_method: peaks
    tom_relative_naming: order
    tom_relative_min_gap: 1.5
    cymbal_crash_weight: 0.7
    stems:
      toms: toms
      crash: crash
      ride: ride
    tom_relative_smooth: 0.5
    tom_relative_min_frac: 0.0
    tom_f0_method: median
- name: order_gap15_sm050_fr05_peak
  rules:
    window_ms: 120
    cymbal_window_ms: 80
    tom_fmin: 50
    tom_fmax: 400
    tom_bands:
    - 113
    - 142
    tom_gate_ratio: 0.1
    tom_peak_mode: raw
    tom_mode: relative
    tom_relative_method: peaks
    tom_relative_naming: order
    tom_relative_min_gap: 1.5
    cymbal_crash_weight: 0.7
    stems:
      toms: toms
      crash: crash
      ride: ride
    tom_relative_smooth: 0.5
    tom_relative_min_frac: 0.05
    tom_f0_method: peak
- name: order_gap15_sm050_fr05_median
  rules:
    window_ms: 120
    cymbal_window_ms: 80
    tom_fmin: 50
    tom_fmax: 400
    tom_bands:
    - 113
    - 142
    tom_gate_ratio: 0.1
    tom_peak_mode: raw
    tom_mode: relative
    tom_relative_method: peaks
    tom_relative_naming: order
    tom_relative_min_gap: 1.5
    cymbal_crash_weight: 0.7
    stems:
      toms: toms
      crash: crash
      ride: ride
    tom_relative_smooth: 0.5
    tom_relative_min_frac: 0.05
    tom_f0_method: median
- name: order_gap15_sm050_fr10_peak
  rules:
    window_ms: 120
    cymbal_window_ms: 80
    tom_fmin: 50
    tom_fmax: 400
    tom_bands:
    - 113
    - 142
    tom_gate_ratio: 0.1
    tom_peak_mode: raw
    tom_mode: relative
    tom_relative_method: peaks
    tom_relative_naming: order
    tom_relative_min_gap: 1.5
    cymbal_crash_weight: 0.7
    stems:
      toms: toms
      crash: crash
      ride: ride
    tom_relative_smooth: 0.5
    tom_relative_min_frac: 0.1
    tom_f0_method: peak
- name: order_gap15_sm050_fr10_median
  rules:
    window_ms: 120
    cymbal_window_ms: 80
    tom_fmin: 50
    tom_fmax: 400
    tom_bands:
    - 113
    - 142
    tom_gate_ratio: 0.1
    tom_peak_mode: raw
    tom_mode: relative
    tom_relative_method: peaks
    tom_relative_naming: order
    tom_relative_min_gap: 1.5
    cymbal_crash_weight: 0.7
    stems:
      toms: toms
      crash: crash
      ride: ride
    tom_relative_smooth: 0.5
    tom_relative_min_frac: 0.1
    tom_f0_method: median
- name: order_gap15_sm075_fr00_peak
  rules:
    window_ms: 120
    cymbal_window_ms: 80
    tom_fmin: 50
    tom_fmax: 400
    tom_bands:
    - 113
    - 142
    tom_gate_ratio: 0.1
    tom_peak_mode: raw
    tom_mode: relative
    tom_relative_method: peaks
    tom_relative_naming: order
    tom_relative_min_gap: 1.5
    cymbal_crash_weight: 0.7
    stems:
      toms: toms
      crash: crash
      ride: ride
    tom_relative_smooth: 0.75
    tom_relative_min_frac: 0.0
    tom_f0_method: peak
- name: order_gap15_sm075_fr00_median
  rules:
    window_ms: 120
    cymbal_window_ms: 80
    tom_fmin: 50
    tom_fmax: 400
    tom_bands:
    - 113
    - 142
    tom_gate_ratio: 0.1
    tom_peak_mode: raw
    tom_mode: relative
    tom_relative_method: peaks
    tom_relative_naming: order
    tom_relative_min_gap: 1.5
    cymbal_crash_weight: 0.7
    stems:
      toms: toms
      crash: crash
      ride: ride
    tom_relative_smooth: 0.75
    tom_relative_min_frac: 0.0
    tom_f0_method: median
- name: order_gap15_sm075_fr05_peak
  rules:
    window_ms: 120
    cymbal_window_ms: 80
    tom_fmin: 50
    tom_fmax: 400
    tom_bands:
    - 113
    - 142
    tom_gate_ratio: 0.1
    tom_peak_mode: raw
    tom_mode: relative
    tom_relative_method: peaks
    tom_relative_naming: order
    tom_relative_min_gap: 1.5
    cymbal_crash_weight: 0.7
    stems:
      toms: toms
      crash: crash
      ride: ride
    tom_relative_smooth: 0.75
    tom_relative_min_frac: 0.05
    tom_f0_method: peak
- name: order_gap15_sm075_fr05_median
  rules:
    window_ms: 120
    cymbal_window_ms: 80
    tom_fmin: 50
    tom_fmax: 400
    tom_bands:
    - 113
    - 142
    tom_gate_ratio: 0.1
    tom_peak_mode: raw
    tom_mode: relative
    tom_relative_method: peaks
    tom_relative_naming: order
    tom_relative_min_gap: 1.5
    cymbal_crash_weight: 0.7
    stems:
      toms: toms
      crash: crash
      ride: ride
    tom_relative_smooth: 0.75
    tom_relative_min_frac: 0.05
    tom_f0_method: median
- name: order_gap15_sm075_fr10_peak
  rules:
    window_ms: 120
    cymbal_window_ms: 80
    tom_fmin: 50
    tom_fmax: 400
    tom_bands:
    - 113
    - 142
    tom_gate_ratio: 0.1
    tom_peak_mode: raw
    tom_mode: relative
    tom_relative_method: peaks
    tom_relative_naming: order
    tom_relative_min_gap: 1.5
    cymbal_crash_weight: 0.7
    stems:
      toms: toms
      crash: crash
      ride: ride
    tom_relative_smooth: 0.75
    tom_relative_min_frac: 0.1
    tom_f0_method: peak
- name: order_gap15_sm075_fr10_median
  rules:
    window_ms: 120
    cymbal_window_ms: 80
    tom_fmin: 50
    tom_fmax: 400
    tom_bands:
    - 113
    - 142
    tom_gate_ratio: 0.1
    tom_peak_mode: raw
    tom_mode: relative
    tom_relative_method: peaks
    tom_relative_naming: order
    tom_relative_min_gap: 1.5
    cymbal_crash_weight: 0.7
    stems:
      toms: toms
      crash: crash
      ride: ride
    tom_relative_smooth: 0.75
    tom_relative_min_frac: 0.1
    tom_f0_method: median
- name: order_gap20_sm025_fr00_peak
  rules:
    window_ms: 120
    cymbal_window_ms: 80
    tom_fmin: 50
    tom_fmax: 400
    tom_bands:
    - 113
    - 142
    tom_gate_ratio: 0.1
    tom_peak_mode: raw
    tom_mode: relative
    tom_relative_method: peaks
    tom_relative_naming: order
    tom_relative_min_gap: 2.0
    cymbal_crash_weight: 0.7
    stems:
      toms: toms
      crash: crash
      ride: ride
    tom_relative_smooth: 0.25
    tom_relative_min_frac: 0.0
    tom_f0_method: peak
- name: order_gap20_sm025_fr00_median
  rules:
    window_ms: 120
    cymbal_window_ms: 80
    tom_fmin: 50
    tom_fmax: 400
    tom_bands:
    - 113
    - 142
    tom_gate_ratio: 0.1
    tom_peak_mode: raw
    tom_mode: relative
    tom_relative_method: peaks
    tom_relative_naming: order
    tom_relative_min_gap: 2.0
    cymbal_crash_weight: 0.7
    stems:
      toms: toms
      crash: crash
      ride: ride
    tom_relative_smooth: 0.25
    tom_relative_min_frac: 0.0
    tom_f0_method: median
- name: order_gap20_sm025_fr05_peak
  rules:
    window_ms: 120
    cymbal_window_ms: 80
    tom_fmin: 50
    tom_fmax: 400
    tom_bands:
    - 113
    - 142
    tom_gate_ratio: 0.1
    tom_peak_mode: raw
    tom_mode: relative
    tom_relative_method: peaks
    tom_relative_naming: order
    tom_relative_min_gap: 2.0
    cymbal_crash_weight: 0.7
    stems:
      toms: toms
      crash: crash
      ride: ride
    tom_relative_smooth: 0.25
    tom_relative_min_frac: 0.05
    tom_f0_method: peak
- name: order_gap20_sm025_fr05_median
  rules:
    window_ms: 120
    cymbal_window_ms: 80
    tom_fmin: 50
    tom_fmax: 400
    tom_bands:
    - 113
    - 142
    tom_gate_ratio: 0.1
    tom_peak_mode: raw
    tom_mode: relative
    tom_relative_method: peaks
    tom_relative_naming: order
    tom_relative_min_gap: 2.0
    cymbal_crash_weight: 0.7
    stems:
      toms: toms
      crash: crash
      ride: ride
    tom_relative_smooth: 0.25
    tom_relative_min_frac: 0.05
    tom_f0_method: median
- name: order_gap20_sm025_fr10_peak
  rules:
    window_ms: 120
    cymbal_window_ms: 80
    tom_fmin: 50
    tom_fmax: 400
    tom_bands:
    - 113
    - 142
    tom_gate_ratio: 0.1
    tom_peak_mode: raw
    tom_mode: relative
    tom_relative_method: peaks
    tom_relative_naming: order
    tom_relative_min_gap: 2.0
    cymbal_crash_weight: 0.7
    stems:
      toms: toms
      crash: crash
      ride: ride
    tom_relative_smooth: 0.25
    tom_relative_min_frac: 0.1
    tom_f0_method: peak
- name: order_gap20_sm025_fr10_median
  rules:
    window_ms: 120
    cymbal_window_ms: 80
    tom_fmin: 50
    tom_fmax: 400
    tom_bands:
    - 113
    - 142
    tom_gate_ratio: 0.1
    tom_peak_mode: raw
    tom_mode: relative
    tom_relative_method: peaks
    tom_relative_naming: order
    tom_relative_min_gap: 2.0
    cymbal_crash_weight: 0.7
    stems:
      toms: toms
      crash: crash
      ride: ride
    tom_relative_smooth: 0.25
    tom_relative_min_frac: 0.1
    tom_f0_method: median
- name: order_gap20_sm050_fr00_peak
  rules:
    window_ms: 120
    cymbal_window_ms: 80
    tom_fmin: 50
    tom_fmax: 400
    tom_bands:
    - 113
    - 142
    tom_gate_ratio: 0.1
    tom_peak_mode: raw
    tom_mode: relative
    tom_relative_method: peaks
    tom_relative_naming: order
    tom_relative_min_gap: 2.0
    cymbal_crash_weight: 0.7
    stems:
      toms: toms
      crash: crash
      ride: ride
    tom_relative_smooth: 0.5
    tom_relative_min_frac: 0.0
    tom_f0_method: peak
- name: order_gap20_sm050_fr00_median
  rules:
    window_ms: 120
    cymbal_window_ms: 80
    tom_fmin: 50
    tom_fmax: 400
    tom_bands:
    - 113
    - 142
    tom_gate_ratio: 0.1
    tom_peak_mode: raw
    tom_mode: relative
    tom_relative_method: peaks
    tom_relative_naming: order
    tom_relative_min_gap: 2.0
    cymbal_crash_weight: 0.7
    stems:
      toms: toms
      crash: crash
      ride: ride
    tom_relative_smooth: 0.5
    tom_relative_min_frac: 0.0
    tom_f0_method: median
- name: order_gap20_sm050_fr05_peak
  rules:
    window_ms: 120
    cymbal_window_ms: 80
    tom_fmin: 50
    tom_fmax: 400
    tom_bands:
    - 113
    - 142
    tom_gate_ratio: 0.1
    tom_peak_mode: raw
    tom_mode: relative
    tom_relative_method: peaks
    tom_relative_naming: order
    tom_relative_min_gap: 2.0
    cymbal_crash_weight: 0.7
    stems:
      toms: toms
      crash: crash
      ride: ride
    tom_relative_smooth: 0.5
    tom_relative_min_frac: 0.05
    tom_f0_method: peak
- name: order_gap20_sm050_fr05_median
  rules:
    window_ms: 120
    cymbal_window_ms: 80
    tom_fmin: 50
    tom_fmax: 400
    tom_bands:
    - 113
    - 142
    tom_gate_ratio: 0.1
    tom_peak_mode: raw
    tom_mode: relative
    tom_relative_method: peaks
    tom_relative_naming: order
    tom_relative_min_gap: 2.0
    cymbal_crash_weight: 0.7
    stems:
      toms: toms
      crash: crash
      ride: ride
    tom_relative_smooth: 0.5
    tom_relative_min_frac: 0.05
    tom_f0_method: median
- name: order_gap20_sm050_fr10_peak
  rules:
    window_ms: 120
    cymbal_window_ms: 80
    tom_fmin: 50
    tom_fmax: 400
    tom_bands:
    - 113
    - 142
    tom_gate_ratio: 0.1
    tom_peak_mode: raw
    tom_mode: relative
    tom_relative_method: peaks
    tom_relative_naming: order
    tom_relative_min_gap: 2.0
    cymbal_crash_weight: 0.7
    stems:
      toms: toms
      crash: crash
      ride: ride
    tom_relative_smooth: 0.5
    tom_relative_min_frac: 0.1
    tom_f0_method: peak
- name: order_gap20_sm050_fr10_median
  rules:
    window_ms: 120
    cymbal_window_ms: 80
    tom_fmin: 50
    tom_fmax: 400
    tom_bands:
    - 113
    - 142
    tom_gate_ratio: 0.1
    tom_peak_mode: raw
    tom_mode: relative
    tom_relative_method: peaks
    tom_relative_naming: order
    tom_relative_min_gap: 2.0
    cymbal_crash_weight: 0.7
    stems:
      toms: toms
      crash: crash
      ride: ride
    tom_relative_smooth: 0.5
    tom_relative_min_frac: 0.1
    tom_f0_method: median
- name: order_gap20_sm075_fr00_peak
  rules:
    window_ms: 120
    cymbal_window_ms: 80
    tom_fmin: 50
    tom_fmax: 400
    tom_bands:
    - 113
    - 142
    tom_gate_ratio: 0.1
    tom_peak_mode: raw
    tom_mode: relative
    tom_relative_method: peaks
    tom_relative_naming: order
    tom_relative_min_gap: 2.0
    cymbal_crash_weight: 0.7
    stems:
      toms: toms
      crash: crash
      ride: ride
    tom_relative_smooth: 0.75
    tom_relative_min_frac: 0.0
    tom_f0_method: peak
- name: order_gap20_sm075_fr00_median
  rules:
    window_ms: 120
    cymbal_window_ms: 80
    tom_fmin: 50
    tom_fmax: 400
    tom_bands:
    - 113
    - 142
    tom_gate_ratio: 0.1
    tom_peak_mode: raw
    tom_mode: relative
    tom_relative_method: peaks
    tom_relative_naming: order
    tom_relative_min_gap: 2.0
    cymbal_crash_weight: 0.7
    stems:
      toms: toms
      crash: crash
      ride: ride
    tom_relative_smooth: 0.75
    tom_relative_min_frac: 0.0
    tom_f0_method: median
- name: order_gap20_sm075_fr05_peak
  rules:
    window_ms: 120
    cymbal_window_ms: 80
    tom_fmin: 50
    tom_fmax: 400
    tom_bands:
    - 113
    - 142
    tom_gate_ratio: 0.1
    tom_peak_mode: raw
    tom_mode: relative
    tom_relative_method: peaks
    tom_relative_naming: order
    tom_relative_min_gap: 2.0
    cymbal_crash_weight: 0.7
    stems:
      toms: toms
      crash: crash
      ride: ride
    tom_relative_smooth: 0.75
    tom_relative_min_frac: 0.05
    tom_f0_method: peak
- name: order_gap20_sm075_fr05_median
  rules:
    window_ms: 120
    cymbal_window_ms: 80
    tom_fmin: 50
    tom_fmax: 400
    tom_bands:
    - 113
    - 142
    tom_gate_ratio: 0.1
    tom_peak_mode: raw
    tom_mode: relative
    tom_relative_method: peaks
    tom_relative_naming: order
    tom_relative_min_gap: 2.0
    cymbal_crash_weight: 0.7
    stems:
      toms: toms
      crash: crash
      ride: ride
    tom_relative_smooth: 0.75
    tom_relative_min_frac: 0.05
    tom_f0_method: median
- name: order_gap20_sm075_fr10_peak
  rules:
    window_ms: 120
    cymbal_window_ms: 80
    tom_fmin: 50
    tom_fmax: 400
    tom_bands:
    - 113
    - 142
    tom_gate_ratio: 0.1
    tom_peak_mode: raw
    tom_mode: relative
    tom_relative_method: peaks
    tom_relative_naming: order
    tom_relative_min_gap: 2.0
    cymbal_crash_weight: 0.7
    stems:
      toms: toms
      crash: crash
      ride: ride
    tom_relative_smooth: 0.75
    tom_relative_min_frac: 0.1
    tom_f0_method: peak
- name: order_gap20_sm075_fr10_median
  rules:
    window_ms: 120
    cymbal_window_ms: 80
    tom_fmin: 50
    tom_fmax: 400
    tom_bands:
    - 113
    - 142
    tom_gate_ratio: 0.1
    tom_peak_mode: raw
    tom_mode: relative
    tom_relative_method: peaks
    tom_relative_naming: order
    tom_relative_min_gap: 2.0
    cymbal_crash_weight: 0.7
    stems:
      toms: toms
      crash: crash
      ride: ride
    tom_relative_smooth: 0.75
    tom_relative_min_frac: 0.1
    tom_f0_method: median
- name: order_gap25_sm025_fr00_peak
  rules:
    window_ms: 120
    cymbal_window_ms: 80
    tom_fmin: 50
    tom_fmax: 400
    tom_bands:
    - 113
    - 142
    tom_gate_ratio: 0.1
    tom_peak_mode: raw
    tom_mode: relative
    tom_relative_method: peaks
    tom_relative_naming: order
    tom_relative_min_gap: 2.5
    cymbal_crash_weight: 0.7
    stems:
      toms: toms
      crash: crash
      ride: ride
    tom_relative_smooth: 0.25
    tom_relative_min_frac: 0.0
    tom_f0_method: peak
- name: order_gap25_sm025_fr00_median
  rules:
    window_ms: 120
    cymbal_window_ms: 80
    tom_fmin: 50
    tom_fmax: 400
    tom_bands:
    - 113
    - 142
    tom_gate_ratio: 0.1
    tom_peak_mode: raw
    tom_mode: relative
    tom_relative_method: peaks
    tom_relative_naming: order
    tom_relative_min_gap: 2.5
    cymbal_crash_weight: 0.7
    stems:
      toms: toms
      crash: crash
      ride: ride
    tom_relative_smooth: 0.25
    tom_relative_min_frac: 0.0
    tom_f0_method: median
- name: order_gap25_sm025_fr05_peak
  rules:
    window_ms: 120
    cymbal_window_ms: 80
    tom_fmin: 50
    tom_fmax: 400
    tom_bands:
    - 113
    - 142
    tom_gate_ratio: 0.1
    tom_peak_mode: raw
    tom_mode: relative
    tom_relative_method: peaks
    tom_relative_naming: order
    tom_relative_min_gap: 2.5
    cymbal_crash_weight: 0.7
    stems:
      toms: toms
      crash: crash
      ride: ride
    tom_relative_smooth: 0.25
    tom_relative_min_frac: 0.05
    tom_f0_method: peak
- name: order_gap25_sm025_fr05_median
  rules:
    window_ms: 120
    cymbal_window_ms: 80
    tom_fmin: 50
    tom_fmax: 400
    tom_bands:
    - 113
    - 142
    tom_gate_ratio: 0.1
    tom_peak_mode: raw
    tom_mode: relative
    tom_relative_method: peaks
    tom_relative_naming: order
    tom_relative_min_gap: 2.5
    cymbal_crash_weight: 0.7
    stems:
      toms: toms
      crash: crash
      ride: ride
    tom_relative_smooth: 0.25
    tom_relative_min_frac: 0.05
    tom_f0_method: median
- name: order_gap25_sm025_fr10_peak
  rules:
    window_ms: 120
    cymbal_window_ms: 80
    tom_fmin: 50
    tom_fmax: 400
    tom_bands:
    - 113
    - 142
    tom_gate_ratio: 0.1
    tom_peak_mode: raw
    tom_mode: relative
    tom_relative_method: peaks
    tom_relative_naming: order
    tom_relative_min_gap: 2.5
    cymbal_crash_weight: 0.7
    stems:
      toms: toms
      crash: crash
      ride: ride
    tom_relative_smooth: 0.25
    tom_relative_min_frac: 0.1
    tom_f0_method: peak
- name: order_gap25_sm025_fr10_median
  rules:
    window_ms: 120
    cymbal_window_ms: 80
    tom_fmin: 50
    tom_fmax: 400
    tom_bands:
    - 113
    - 142
    tom_gate_ratio: 0.1
    tom_peak_mode: raw
    tom_mode: relative
    tom_relative_method: peaks
    tom_relative_naming: order
    tom_relative_min_gap: 2.5
    cymbal_crash_weight: 0.7
    stems:
      toms: toms
      crash: crash
      ride: ride
    tom_relative_smooth: 0.25
    tom_relative_min_frac: 0.1
    tom_f0_method: median
- name: order_gap25_sm050_fr00_peak
  rules:
    window_ms: 120
    cymbal_window_ms: 80
    tom_fmin: 50
    tom_fmax: 400
    tom_bands:
    - 113
    - 142
    tom_gate_ratio: 0.1
    tom_peak_mode: raw
    tom_mode: relative
    tom_relative_method: peaks
    tom_relative_naming: order
    tom_relative_min_gap: 2.5
    cymbal_crash_weight: 0.7
    stems:
      toms: toms
      crash: crash
      ride: ride
    tom_relative_smooth: 0.5
    tom_relative_min_frac: 0.0
    tom_f0_method: peak
- name: order_gap25_sm050_fr00_median
  rules:
    window_ms: 120
    cymbal_window_ms: 80
    tom_fmin: 50
    tom_fmax: 400
    tom_bands:
    - 113
    - 142
    tom_gate_ratio: 0.1
    tom_peak_mode: raw
    tom_mode: relative
    tom_relative_method: peaks
    tom_relative_naming: order
    tom_relative_min_gap: 2.5
    cymbal_crash_weight: 0.7
    stems:
      toms: toms
      crash: crash
      ride: ride
    tom_relative_smooth: 0.5
    tom_relative_min_frac: 0.0
    tom_f0_method: median
- name: order_gap25_sm050_fr05_peak
  rules:
    window_ms: 120
    cymbal_window_ms: 80
    tom_fmin: 50
    tom_fmax: 400
    tom_bands:
    - 113
    - 142
    tom_gate_ratio: 0.1
    tom_peak_mode: raw
    tom_mode: relative
    tom_relative_method: peaks
    tom_relative_naming: order
    tom_relative_min_gap: 2.5
    cymbal_crash_weight: 0.7
    stems:
      toms: toms
      crash: crash
      ride: ride
    tom_relative_smooth: 0.5
    tom_relative_min_frac: 0.05
    tom_f0_method: peak
- name: order_gap25_sm050_fr05_median
  rules:
    window_ms: 120
    cymbal_window_ms: 80
    tom_fmin: 50
    tom_fmax: 400
    tom_bands:
    - 113
    - 142
    tom_gate_ratio: 0.1
    tom_peak_mode: raw
    tom_mode: relative
    tom_relative_method: peaks
    tom_relative_naming: order
    tom_relative_min_gap: 2.5
    cymbal_crash_weight: 0.7
    stems:
      toms: toms
      crash: crash
      ride: ride
    tom_relative_smooth: 0.5
    tom_relative_min_frac: 0.05
    tom_f0_method: median
- name: order_gap25_sm050_fr10_peak
  rules:
    window_ms: 120
    cymbal_window_ms: 80
    tom_fmin: 50
    tom_fmax: 400
    tom_bands:
    - 113
    - 142
    tom_gate_ratio: 0.1
    tom_peak_mode: raw
    tom_mode: relative
    tom_relative_method: peaks
    tom_relative_naming: order
    tom_relative_min_gap: 2.5
    cymbal_crash_weight: 0.7
    stems:
      toms: toms
      crash: crash
      ride: ride
    tom_relative_smooth: 0.5
    tom_relative_min_frac: 0.1
    tom_f0_method: peak
- name: order_gap25_sm050_fr10_median
  rules:
    window_ms: 120
    cymbal_window_ms: 80
    tom_fmin: 50
    tom_fmax: 400
    tom_bands:
    - 113
    - 142
    tom_gate_ratio: 0.1
    tom_peak_mode: raw
    tom_mode: relative
    tom_relative_method: peaks
    tom_relative_naming: order
    tom_relative_min_gap: 2.5
    cymbal_crash_weight: 0.7
    stems:
      toms: toms
      crash: crash
      ride: ride
    tom_relative_smooth: 0.5
    tom_relative_min_frac: 0.1
    tom_f0_method: median
- name: order_gap25_sm075_fr00_peak
  rules:
    window_ms: 120
    cymbal_window_ms: 80
    tom_fmin: 50
    tom_fmax: 400
    tom_bands:
    - 113
    - 142
    tom_gate_ratio: 0.1
    tom_peak_mode: raw
    tom_mode: relative
    tom_relative_method: peaks
    tom_relative_naming: order
    tom_relative_min_gap: 2.5
    cymbal_crash_weight: 0.7
    stems:
      toms: toms
      crash: crash
      ride: ride
    tom_relative_smooth: 0.75
    tom_relative_min_frac: 0.0
    tom_f0_method: peak
- name: order_gap25_sm075_fr00_median
  rules:
    window_ms: 120
    cymbal_window_ms: 80
    tom_fmin: 50
    tom_fmax: 400
    tom_bands:
    - 113
    - 142
    tom_gate_ratio: 0.1
    tom_peak_mode: raw
    tom_mode: relative
    tom_relative_method: peaks
    tom_relative_naming: order
    tom_relative_min_gap: 2.5
    cymbal_crash_weight: 0.7
    stems:
      toms: toms
      crash: crash
      ride: ride
    tom_relative_smooth: 0.75
    tom_relative_min_frac: 0.0
    tom_f0_method: median
- name: order_gap25_sm075_fr05_peak
  rules:
    window_ms: 120
    cymbal_window_ms: 80
    tom_fmin: 50
    tom_fmax: 400
    tom_bands:
    - 113
    - 142
    tom_gate_ratio: 0.1
    tom_peak_mode: raw
    tom_mode: relative
    tom_relative_method: peaks
    tom_relative_naming: order
    tom_relative_min_gap: 2.5
    cymbal_crash_weight: 0.7
    stems:
      toms: toms
      crash: crash
      ride: ride
    tom_relative_smooth: 0.75
    tom_relative_min_frac: 0.05
    tom_f0_method: peak
- name: order_gap25_sm075_fr05_median
  rules:
    window_ms: 120
    cymbal_window_ms: 80
    tom_fmin: 50
    tom_fmax: 400
    tom_bands:
    - 113
    - 142
    tom_gate_ratio: 0.1
    tom_peak_mode: raw
    tom_mode: relative
    tom_relative_method: peaks
    tom_relative_naming: order
    tom_relative_min_gap: 2.5
    cymbal_crash_weight: 0.7
    stems:
      toms: toms
      crash: crash
      ride: ride
    tom_relative_smooth: 0.75
    tom_relative_min_frac: 0.05
    tom_f0_method: median
- name: order_gap25_sm075_fr10_peak
  rules:
    window_ms: 120
    cymbal_window_ms: 80
    tom_fmin: 50
    tom_fmax: 400
    tom_bands:
    - 113
    - 142
    tom_gate_ratio: 0.1
    tom_peak_mode: raw
    tom_mode: relative
    tom_relative_method: peaks
    tom_relative_naming: order
    tom_relative_min_gap: 2.5
    cymbal_crash_weight: 0.7
    stems:
      toms: toms
      crash: crash
      ride: ride
    tom_relative_smooth: 0.75
    tom_relative_min_frac: 0.1
    tom_f0_method: peak
- name: order_gap25_sm075_fr10_median
  rules:
    window_ms: 120
    cymbal_window_ms: 80
    tom_fmin: 50
    tom_fmax: 400
    tom_bands:
    - 113
    - 142
    tom_gate_ratio: 0.1
    tom_peak_mode: raw
    tom_mode: relative
    tom_relative_method: peaks
    tom_relative_naming: order
    tom_relative_min_gap: 2.5
    cymbal_crash_weight: 0.7
    stems:
      toms: toms
      crash: crash
      ride: ride
    tom_relative_smooth: 0.75
    tom_relative_min_frac: 0.1
    tom_f0_method: median
- name: order_gap30_sm025_fr00_peak
  rules:
    window_ms: 120
    cymbal_window_ms: 80
    tom_fmin: 50
    tom_fmax: 400
    tom_bands:
    - 113
    - 142
    tom_gate_ratio: 0.1
    tom_peak_mode: raw
    tom_mode: relative
    tom_relative_method: peaks
    tom_relative_naming: order
    tom_relative_min_gap: 3.0
    cymbal_crash_weight: 0.7
    stems:
      toms: toms
      crash: crash
      ride: ride
    tom_relative_smooth: 0.25
    tom_relative_min_frac: 0.0
    tom_f0_method: peak
- name: order_gap30_sm025_fr00_median
  rules:
    window_ms: 120
    cymbal_window_ms: 80
    tom_fmin: 50
    tom_fmax: 400
    tom_bands:
    - 113
    - 142
    tom_gate_ratio: 0.1
    tom_peak_mode: raw
    tom_mode: relative
    tom_relative_method: peaks
    tom_relative_naming: order
    tom_relative_min_gap: 3.0
    cymbal_crash_weight: 0.7
    stems:
      toms: toms
      crash: crash
      ride: ride
    tom_relative_smooth: 0.25
    tom_relative_min_frac: 0.0
    tom_f0_method: median
- name: order_gap30_sm025_fr05_peak
  rules:
    window_ms: 120
    cymbal_window_ms: 80
    tom_fmin: 50
    tom_fmax: 400
    tom_bands:
    - 113
    - 142
    tom_gate_ratio: 0.1
    tom_peak_mode: raw
    tom_mode: relative
    tom_relative_method: peaks
    tom_relative_naming: order
    tom_relative_min_gap: 3.0
    cymbal_crash_weight: 0.7
    stems:
      toms: toms
      crash: crash
      ride: ride
    tom_relative_smooth: 0.25
    tom_relative_min_frac: 0.05
    tom_f0_method: peak
- name: order_gap30_sm025_fr05_median
  rules:
    window_ms: 120
    cymbal_window_ms: 80
    tom_fmin: 50
    tom_fmax: 400
    tom_bands:
    - 113
    - 142
    tom_gate_ratio: 0.1
    tom_peak_mode: raw
    tom_mode: relative
    tom_relative_method: peaks
    tom_relative_naming: order
    tom_relative_min_gap: 3.0
    cymbal_crash_weight: 0.7
    stems:
      toms: toms
      crash: crash
      ride: ride
    tom_relative_smooth: 0.25
    tom_relative_min_frac: 0.05
    tom_f0_method: median
- name: order_gap30_sm025_fr10_peak
  rules:
    window_ms: 120
    cymbal_window_ms: 80
    tom_fmin: 50
    tom_fmax: 400
    tom_bands:
    - 113
    - 142
    tom_gate_ratio: 0.1
    tom_peak_mode: raw
    tom_mode: relative
    tom_relative_method: peaks
    tom_relative_naming: order
    tom_relative_min_gap: 3.0
    cymbal_crash_weight: 0.7
    stems:
      toms: toms
      crash: crash
      ride: ride
    tom_relative_smooth: 0.25
    tom_relative_min_frac: 0.1
    tom_f0_method: peak
- name: order_gap30_sm025_fr10_median
  rules:
    window_ms: 120
    cymbal_window_ms: 80
    tom_fmin: 50
    tom_fmax: 400
    tom_bands:
    - 113
    - 142
    tom_gate_ratio: 0.1
    tom_peak_mode: raw
    tom_mode: relative
    tom_relative_method: peaks
    tom_relative_naming: order
    tom_relative_min_gap: 3.0
    cymbal_crash_weight: 0.7
    stems:
      toms: toms
      crash: crash
      ride: ride
    tom_relative_smooth: 0.25
    tom_relative_min_frac: 0.1
    tom_f0_method: median
- name: order_gap30_sm050_fr00_peak
  rules:
    window_ms: 120
    cymbal_window_ms: 80
    tom_fmin: 50
    tom_fmax: 400
    tom_bands:
    - 113
    - 142
    tom_gate_ratio: 0.1
    tom_peak_mode: raw
    tom_mode: relative
    tom_relative_method: peaks
    tom_relative_naming: order
    tom_relative_min_gap: 3.0
    cymbal_crash_weight: 0.7
    stems:
      toms: toms
      crash: crash
      ride: ride
    tom_relative_smooth: 0.5
    tom_relative_min_frac: 0.0
    tom_f0_method: peak
- name: order_gap30_sm050_fr00_median
  rules:
    window_ms: 120
    cymbal_window_ms: 80
    tom_fmin: 50
    tom_fmax: 400
    tom_bands:
    - 113
    - 142
    tom_gate_ratio: 0.1
    tom_peak_mode: raw
    tom_mode: relative
    tom_relative_method: peaks
    tom_relative_naming: order
    tom_relative_min_gap: 3.0
    cymbal_crash_weight: 0.7
    stems:
      toms: toms
      crash: crash
      ride: ride
    tom_relative_smooth: 0.5
    tom_relative_min_frac: 0.0
    tom_f0_method: median
- name: order_gap30_sm050_fr05_peak
  rules:
    window_ms: 120
    cymbal_window_ms: 80
    tom_fmin: 50
    tom_fmax: 400
    tom_bands:
    - 113
    - 142
    tom_gate_ratio: 0.1
    tom_peak_mode: raw
    tom_mode: relative
    tom_relative_method: peaks
    tom_relative_naming: order
    tom_relative_min_gap: 3.0
    cymbal_crash_weight: 0.7
    stems:
      toms: toms
      crash: crash
      ride: ride
    tom_relative_smooth: 0.5
    tom_relative_min_frac: 0.05
    tom_f0_method: peak
- name: order_gap30_sm050_fr05_median
  rules:
    window_ms: 120
    cymbal_window_ms: 80
    tom_fmin: 50
    tom_fmax: 400
    tom_bands:
    - 113
    - 142
    tom_gate_ratio: 0.1
    tom_peak_mode: raw
    tom_mode: relative
    tom_relative_method: peaks
    tom_relative_naming: order
    tom_relative_min_gap: 3.0
    cymbal_crash_weight: 0.7
    stems:
      toms: toms
      crash: crash
      ride: ride
    tom_relative_smooth: 0.5
    tom_relative_min_frac: 0.05
    tom_f0_method: median
- name: order_gap30_sm050_fr10_peak
  rules:
    window_ms: 120
    cymbal_window_ms: 80
    tom_fmin: 50
    tom_fmax: 400
    tom_bands:
    - 113
    - 142
    tom_gate_ratio: 0.1
    tom_peak_mode: raw
    tom_mode: relative
    tom_relative_method: peaks
    tom_relative_naming: order
    tom_relative_min_gap: 3.0
    cymbal_crash_weight: 0.7
    stems:
      toms: toms
      crash: crash
      ride: ride
    tom_relative_smooth: 0.5
    tom_relative_min_frac: 0.1
    tom_f0_method: peak
- name: order_gap30_sm050_fr10_median
  rules:
    window_ms: 120
    cymbal_window_ms: 80
    tom_fmin: 50
    tom_fmax: 400
    tom_bands:
    - 113
    - 142
    tom_gate_ratio: 0.1
    tom_peak_mode: raw
    tom_mode: relative
    tom_relative_method: peaks
    tom_relative_naming: order
    tom_relative_min_gap: 3.0
    cymbal_crash_weight: 0.7
    stems:
      toms: toms
      crash: crash
      ride: ride
    tom_relative_smooth: 0.5
    tom_relative_min_frac: 0.1
    tom_f0_method: median
- name: order_gap30_sm075_fr00_peak
  rules:
    window_ms: 120
    cymbal_window_ms: 80
    tom_fmin: 50
    tom_fmax: 400
    tom_bands:
    - 113
    - 142
    tom_gate_ratio: 0.1
    tom_peak_mode: raw
    tom_mode: relative
    tom_relative_method: peaks
    tom_relative_naming: order
    tom_relative_min_gap: 3.0
    cymbal_crash_weight: 0.7
    stems:
      toms: toms
      crash: crash
      ride: ride
    tom_relative_smooth: 0.75
    tom_relative_min_frac: 0.0
    tom_f0_method: peak
- name: order_gap30_sm075_fr00_median
  rules:
    window_ms: 120
    cymbal_window_ms: 80
    tom_fmin: 50
    tom_fmax: 400
    tom_bands:
    - 113
    - 142
    tom_gate_ratio: 0.1
    tom_peak_mode: raw
    tom_mode: relative
    tom_relative_method: peaks
    tom_relative_naming: order
    tom_relative_min_gap: 3.0
    cymbal_crash_weight: 0.7
    stems:
      toms: toms
      crash: crash
      ride: ride
    tom_relative_smooth: 0.75
    tom_relative_min_frac: 0.0
    tom_f0_method: median
- name: order_gap30_sm075_fr05_peak
  rules:
    window_ms: 120
    cymbal_window_ms: 80
    tom_fmin: 50
    tom_fmax: 400
    tom_bands:
    - 113
    - 142
    tom_gate_ratio: 0.1
    tom_peak_mode: raw
    tom_mode: relative
    tom_relative_method: peaks
    tom_relative_naming: order
    tom_relative_min_gap: 3.0
    cymbal_crash_weight: 0.7
    stems:
      toms: toms
      crash: crash
      ride: ride
    tom_relative_smooth: 0.75
    tom_relative_min_frac: 0.05
    tom_f0_method: peak
- name: order_gap30_sm075_fr05_median
  rules:
    window_ms: 120
    cymbal_window_ms: 80
    tom_fmin: 50
    tom_fmax: 400
    tom_bands:
    - 113
    - 142
    tom_gate_ratio: 0.1
    tom_peak_mode: raw
    tom_mode: relative
    tom_relative_method: peaks
    tom_relative_naming: order
    tom_relative_min_gap: 3.0
    cymbal_crash_weight: 0.7
    stems:
      toms: toms
      crash: crash
      ride: ride
    tom_relative_smooth: 0.75
    tom_relative_min_frac: 0.05
    tom_f0_method: median
- name: order_gap30_sm075_fr10_peak
  rules:
    window_ms: 120
    cymbal_window_ms: 80
    tom_fmin: 50
    tom_fmax: 400
    tom_bands:
    - 113
    - 142
    tom_gate_ratio: 0.1
    tom_peak_mode: raw
    tom_mode: relative
    tom_relative_method: peaks
    tom_relative_naming: order
    tom_relative_min_gap: 3.0
    cymbal_crash_weight: 0.7
    stems:
      toms: toms
      crash: crash
      ride: ride
    tom_relative_smooth: 0.75
    tom_relative_min_frac: 0.1
    tom_f0_method: peak
- name: order_gap30_sm075_fr10_median
  rules:
    window_ms: 120
    cymbal_window_ms: 80
    tom_fmin: 50
    tom_fmax: 400
    tom_bands:
    - 113
    - 142
    tom_gate_ratio: 0.1
    tom_peak_mode: raw
    tom_mode: relative
    tom_relative_method: peaks
    tom_relative_naming: order
    tom_relative_min_gap: 3.0
    cymbal_crash_weight: 0.7
    stems:
      toms: toms
      crash: crash
      ride: ride
    tom_relative_smooth: 0.75
    tom_relative_min_frac: 0.1
    tom_f0_method: median
- name: order_gap35_sm025_fr00_peak
  rules:
    window_ms: 120
    cymbal_window_ms: 80
    tom_fmin: 50
    tom_fmax: 400
    tom_bands:
    - 113
    - 142
    tom_gate_ratio: 0.1
    tom_peak_mode: raw
    tom_mode: relative
    tom_relative_method: peaks
    tom_relative_naming: order
    tom_relative_min_gap: 3.5
    cymbal_crash_weight: 0.7
    stems:
      toms: toms
      crash: crash
      ride: ride
    tom_relative_smooth: 0.25
    tom_relative_min_frac: 0.0
    tom_f0_method: peak
- name: order_gap35_sm025_fr00_median
  rules:
    window_ms: 120
    cymbal_window_ms: 80
    tom_fmin: 50
    tom_fmax: 400
    tom_bands:
    - 113
    - 142
    tom_gate_ratio: 0.1
    tom_peak_mode: raw
    tom_mode: relative
    tom_relative_method: peaks
    tom_relative_naming: order
    tom_relative_min_gap: 3.5
    cymbal_crash_weight: 0.7
    stems:
      toms: toms
      crash: crash
      ride: ride
    tom_relative_smooth: 0.25
    tom_relative_min_frac: 0.0
    tom_f0_method: median
- name: order_gap35_sm025_fr05_peak
  rules:
    window_ms: 120
    cymbal_window_ms: 80
    tom_fmin: 50
    tom_fmax: 400
    tom_bands:
    - 113
    - 142
    tom_gate_ratio: 0.1
    tom_peak_mode: raw
    tom_mode: relative
    tom_relative_method: peaks
    tom_relative_naming: order
    tom_relative_min_gap: 3.5
    cymbal_crash_weight: 0.7
    stems:
      toms: toms
      crash: crash
      ride: ride
    tom_relative_smooth: 0.25
    tom_relative_min_frac: 0.05
    tom_f0_method: peak
- name: order_gap35_sm025_fr05_median
  rules:
    window_ms: 120
    cymbal_window_ms: 80
    tom_fmin: 50
    tom_fmax: 400
    tom_bands:
    - 113
    - 142
    tom_gate_ratio: 0.1
    tom_peak_mode: raw
    tom_mode: relative
    tom_relative_method: peaks
    tom_relative_naming: order
    tom_relative_min_gap: 3.5
    cymbal_crash_weight: 0.7
    stems:
      toms: toms
      crash: crash
      ride: ride
    tom_relative_smooth: 0.25
    tom_relative_min_frac: 0.05
    tom_f0_method: median
- name: order_gap35_sm025_fr10_peak
  rules:
    window_ms: 120
    cymbal_window_ms: 80
    tom_fmin: 50
    tom_fmax: 400
    tom_bands:
    - 113
    - 142
    tom_gate_ratio: 0.1
    tom_peak_mode: raw
    tom_mode: relative
    tom_relative_method: peaks
    tom_relative_naming: order
    tom_relative_min_gap: 3.5
    cymbal_crash_weight: 0.7
    stems:
      toms: toms
      crash: crash
      ride: ride
    tom_relative_smooth: 0.25
    tom_relative_min_frac: 0.1
    tom_f0_method: peak
- name: order_gap35_sm025_fr10_median
  rules:
    window_ms: 120
    cymbal_window_ms: 80
    tom_fmin: 50
    tom_fmax: 400
    tom_bands:
    - 113
    - 142
    tom_gate_ratio: 0.1
    tom_peak_mode: raw
    tom_mode: relative
    tom_relative_method: peaks
    tom_relative_naming: order
    tom_relative_min_gap: 3.5
    cymbal_crash_weight: 0.7
    stems:
      toms: toms
      crash: crash
      ride: ride
    tom_relative_smooth: 0.25
    tom_relative_min_frac: 0.1
    tom_f0_method: median
- name: order_gap35_sm050_fr00_peak
  rules:
    window_ms: 120
    cymbal_window_ms: 80
    tom_fmin: 50
    tom_fmax: 400
    tom_bands:
    - 113
    - 142
    tom_gate_ratio: 0.1
    tom_peak_mode: raw
    tom_mode: relative
    tom_relative_method: peaks
    tom_relative_naming: order
    tom_relative_min_gap: 3.5
    cymbal_crash_weight: 0.7
    stems:
      toms: toms
      crash: crash
      ride: ride
    tom_relative_smooth: 0.5
    tom_relative_min_frac: 0.0
    tom_f0_method: peak
- name: order_gap35_sm050_fr00_median
  rules:
    window_ms: 120
    cymbal_window_ms: 80
    tom_fmin: 50
    tom_fmax: 400
    tom_bands:
    - 113
    - 142
    tom_gate_ratio: 0.1
    tom_peak_mode: raw
    tom_mode: relative
    tom_relative_method: peaks
    tom_relative_naming: order
    tom_relative_min_gap: 3.5
    cymbal_crash_weight: 0.7
    stems:
      toms: toms
      crash: crash
      ride: ride
    tom_relative_smooth: 0.5
    tom_relative_min_frac: 0.0
    tom_f0_method: median
- name: order_gap35_sm050_fr05_peak
  rules:
    window_ms: 120
    cymbal_window_ms: 80
    tom_fmin: 50
    tom_fmax: 400
    tom_bands:
    - 113
    - 142
    tom_gate_ratio: 0.1
    tom_peak_mode: raw
    tom_mode: relative
    tom_relative_method: peaks
    tom_relative_naming: order
    tom_relative_min_gap: 3.5
    cymbal_crash_weight: 0.7
    stems:
      toms: toms
      crash: crash
      ride: ride
    tom_relative_smooth: 0.5
    tom_relative_min_frac: 0.05
    tom_f0_method: peak
- name: order_gap35_sm050_fr05_median
  rules:
    window_ms: 120
    cymbal_window_ms: 80
    tom_fmin: 50
    tom_fmax: 400
    tom_bands:
    - 113
    - 142
    tom_gate_ratio: 0.1
    tom_peak_mode: raw
    tom_mode: relative
    tom_relative_method: peaks
    tom_relative_naming: order
    tom_relative_min_gap: 3.5
    cymbal_crash_weight: 0.7
    stems:
      toms: toms
      crash: crash
      ride: ride
    tom_relative_smooth: 0.5
    tom_relative_min_frac: 0.05
    tom_f0_method: median
- name: order_gap35_sm050_fr10_peak
  rules:
    window_ms: 120
    cymbal_window_ms: 80
    tom_fmin: 50
    tom_fmax: 400
    tom_bands:
    - 113
    - 142
    tom_gate_ratio: 0.1
    tom_peak_mode: raw
    tom_mode: relative
    tom_relative_method: peaks
    tom_relative_naming: order
    tom_relative_min_gap: 3.5
    cymbal_crash_weight: 0.7
    stems:
      toms: toms
      crash: crash
      ride: ride
    tom_relative_smooth: 0.5
    tom_relative_min_frac: 0.1
    tom_f0_method: peak
- name: order_gap35_sm050_fr10_median
  rules:
    window_ms: 120
    cymbal_window_ms: 80
    tom_fmin: 50
    tom_fmax: 400
    tom_bands:
    - 113
    - 142
    tom_gate_ratio: 0.1
    tom_peak_mode: raw
    tom_mode: relative
    tom_relative_method: peaks
    tom_relative_naming: order
    tom_relative_min_gap: 3.5
    cymbal_crash_weight: 0.7
    stems:
      toms: toms
      crash: crash
      ride: ride
    tom_relative_smooth: 0.5
    tom_relative_min_frac: 0.1
    tom_f0_method: median
- name: order_gap35_sm075_fr00_peak
  rules:
    window_ms: 120
    cymbal_window_ms: 80
    tom_fmin: 50
    tom_fmax: 400
    tom_bands:
    - 113
    - 142
    tom_gate_ratio: 0.1
    tom_peak_mode: raw
    tom_mode: relative
    tom_relative_method: peaks
    tom_relative_naming: order
    tom_relative_min_gap: 3.5
    cymbal_crash_weight: 0.7
    stems:
      toms: toms
      crash: crash
      ride: ride
    tom_relative_smooth: 0.75
    tom_relative_min_frac: 0.0
    tom_f0_method: peak
- name: order_gap35_sm075_fr00_median
  rules:
    window_ms: 120
    cymbal_window_ms: 80
    tom_fmin: 50
    tom_fmax: 400
    tom_bands:
    - 113
    - 142
    tom_gate_ratio: 0.1
    tom_peak_mode: raw
    tom_mode: relative
    tom_relative_method: peaks
    tom_relative_naming: order
    tom_relative_min_gap: 3.5
    cymbal_crash_weight: 0.7
    stems:
      toms: toms
      crash: crash
      ride: ride
    tom_relative_smooth: 0.75
    tom_relative_min_frac: 0.0
    tom_f0_method: median
- name: order_gap35_sm075_fr05_peak
  rules:
    window_ms: 120
    cymbal_window_ms: 80
    tom_fmin: 50
    tom_fmax: 400
    tom_bands:
    - 113
    - 142
    tom_gate_ratio: 0.1
    tom_peak_mode: raw
    tom_mode: relative
    tom_relative_method: peaks
    tom_relative_naming: order
    tom_relative_min_gap: 3.5
    cymbal_crash_weight: 0.7
    stems:
      toms: toms
      crash: crash
      ride: ride
    tom_relative_smooth: 0.75
    tom_relative_min_frac: 0.05
    tom_f0_method: peak
- name: order_gap35_sm075_fr05_median
  rules:
    window_ms: 120
    cymbal_window_ms: 80
    tom_fmin: 50
    tom_fmax: 400
    tom_bands:
    - 113
    - 142
    tom_gate_ratio: 0.1
    tom_peak_mode: raw
    tom_mode: relative
    tom_relative_method: peaks
    tom_relative_naming: order
    tom_relative_min_gap: 3.5
    cymbal_crash_weight: 0.7
    stems:
      toms: toms
      crash: crash
      ride: ride
    tom_relative_smooth: 0.75
    tom_relative_min_frac: 0.05
    tom_f0_method: median
- name: order_gap35_sm075_fr10_peak
  rules:
    window_ms: 120
    cymbal_window_ms: 80
    tom_fmin: 50
    tom_fmax: 400
    tom_bands:
    - 113
    - 142
    tom_gate_ratio: 0.1
    tom_peak_mode: raw
    tom_mode: relative
    tom_relative_method: peaks
    tom_relative_naming: order
    tom_relative_min_gap: 3.5
    cymbal_crash_weight: 0.7
    stems:
      toms: toms
      crash: crash
      ride: ride
    tom_relative_smooth: 0.75
    tom_relative_min_frac: 0.1
    tom_f0_method: peak
- name: order_gap35_sm075_fr10_median
  rules:
    window_ms: 120
    cymbal_window_ms: 80
    tom_fmin: 50
    tom_fmax: 400
    tom_bands:
    - 113
    - 142
    tom_gate_ratio: 0.1
    tom_peak_mode: raw
    tom_mode: relative
    tom_relative_method: peaks
    tom_relative_naming: order
    tom_relative_min_gap: 3.5
    cymbal_crash_weight: 0.7
    stems:
      toms: toms
      crash: crash
      ride: ride
    tom_relative_smooth: 0.75
    tom_relative_min_frac: 0.1
    tom_f0_method: median
''',
    f"{TOOL}/candidates_enst.yaml": r'''
candidates:
- name: current
  rules:
    window_ms: 120
    cymbal_window_ms: 80
    tom_fmin: 50
    tom_fmax: 400
    tom_bands:
    - 113
    - 142
    tom_gate_ratio: 0.1
    tom_peak_mode: raw
    tom_mode: relative
    tom_relative_method: peaks
    tom_relative_naming: hybrid
    tom_relative_min_gap: 3.0
    cymbal_crash_weight: 0.7
    stems:
      toms: toms
      crash: crash
      ride: ride
- name: current_order
  rules:
    window_ms: 120
    cymbal_window_ms: 80
    tom_fmin: 50
    tom_fmax: 400
    tom_bands:
    - 113
    - 142
    tom_gate_ratio: 0.1
    tom_peak_mode: raw
    tom_mode: relative
    tom_relative_method: peaks
    tom_relative_naming: order
    tom_relative_min_gap: 3.0
    cymbal_crash_weight: 0.7
    stems:
      toms: toms
      crash: crash
      ride: ride
- name: order_gap25_sm025_fr05_peak
  rules:
    window_ms: 120
    cymbal_window_ms: 80
    tom_fmin: 50
    tom_fmax: 400
    tom_bands:
    - 113
    - 142
    tom_gate_ratio: 0.1
    tom_peak_mode: raw
    tom_mode: relative
    tom_relative_method: peaks
    tom_relative_naming: order
    tom_relative_min_gap: 2.5
    cymbal_crash_weight: 0.7
    stems:
      toms: toms
      crash: crash
      ride: ride
    tom_relative_smooth: 0.25
    tom_relative_min_frac: 0.05
    tom_f0_method: peak
''',
}
for path, text in FILES.items():
    with open(path, "w", encoding="utf-8") as f:
        f.write(text.lstrip("\n"))
!wc -l $TOOL/tom_rule_eval_stream.py $TOOL/candidates_part1.yaml $TOOL/candidates_enst.yaml
!grep -c "^- name:" $TOOL/candidates_part1.yaml $TOOL/candidates_enst.yaml
# 확인: 파일 3개 줄 수, 후보 수 92 / 3

In [ ]:
# 3. 데이터·특징 점검 (여러 번 쳐도 됨)
!$BENCH check-data $DS/egmd_test_acoustic.yaml $DS/enst_minus_one.yaml $DS/enst_minus_one_gain1.yaml
!for d in egmd_test_acoustic enst_minus_one; do echo "$d: $(ls $FEAT/$d 2>/dev/null | grep -c npz)개"; done
!echo "enst_minus_one_gain1 분리: $(ls $FEAT_SEP/enst_minus_one_gain1 2>/dev/null | grep -c npz)개"
# 확인: check-data가 1,170개·64곡·64곡 누락 0. 특징은 처음이면 0개, 다 만들었으면 1170 / 64 / 64

In [ ]:
# 4. 특징 만들기 (백그라운드, 한 번만). 이미 있는 곡은 건너뜀. 세 개를 차례로 돌림 (GPU 하나라서)
import subprocess

assert not os.path.exists(LOG_FEAT), "이미 시작했음. 5번 셀로 진행을 확인함"
script = " && ".join([
    f"{BENCH} rule-features --dataset {DS}/egmd_test_acoustic.yaml --out {FEAT}",
    f"{BENCH} rule-features --dataset {DS}/enst_minus_one.yaml --out {FEAT}",
    f"{BENCH} rule-features --separator {SEP_MODEL} --dataset {DS}/enst_minus_one_gain1.yaml --out {FEAT_SEP}",
])
proc = subprocess.Popen(["nohup", "nice", "-n", "19", "bash", "-c", script], cwd=AI, stdout=open(LOG_FEAT, "w"),
                        stderr=subprocess.STDOUT, env=dict(os.environ), start_new_session=True)
print("시작. pid", proc.pid, "로그", LOG_FEAT)
# 확인: '시작. pid 숫자'. 로컬 4070에서는 E-GMD가 곡당 7~25초였음. H200이면 더 빠를 것으로 보이나 측정하지 않았음

In [ ]:
# 5. 특징 진행 확인 (여러 번 쳐도 됨)
!for d in egmd_test_acoustic enst_minus_one; do echo "$d: $(ls $FEAT/$d 2>/dev/null | grep -c npz)개"; done
!echo "enst_minus_one_gain1 분리: $(ls $FEAT_SEP/enst_minus_one_gain1 2>/dev/null | grep -c npz)개"
!grep -h "저장 .*실패" $LOG_FEAT || echo "아직 끝난 묶음 없음"
!tail -1 $LOG_FEAT
!ps -u $ME -o pid,etime,args | grep -E "[r]ule-features" | head -1 | cut -c1-120 || true
# 확인: 1170 / 64 / 64가 되고 '저장 N개, 건너뜀 M개, 실패 0개' 줄이 3개 나오면 끝. 실패가 0이 아니면 로그 끝 10줄을 적어 둠

In [ ]:
# 6. 채점 시작 (CPU, 5번이 다 끝난 뒤, 한 번만). E-GMD 후보 92개, ENST 두 묶음 후보 3개
import subprocess

assert not os.path.exists(LOG_EVAL), "이미 시작했음. 7번 셀로 확인함"
run = f"{VPY} {TOOL}/tom_rule_eval_stream.py --two-tom-upper MID_TOM --workers {WORKERS}"
script = " && ".join([
    f"{run} --dataset {DS}/egmd_test_acoustic.yaml --features {FEAT} --candidates {TOOL}/candidates_part1.yaml --out {OUT}/stream_egmd_test.csv",
    f"{run} --dataset {DS}/enst_minus_one.yaml --features {FEAT} --candidates {TOOL}/candidates_enst.yaml --out {OUT}/stream_enst_drums.csv",
    f"{run} --dataset {DS}/enst_minus_one_gain1.yaml --features {FEAT_SEP} --candidates {TOOL}/candidates_enst.yaml --out {OUT}/stream_enst_sep.csv",
])
proc = subprocess.Popen(["nohup", "nice", "-n", "19", "bash", "-c", script], cwd=AI, stdout=open(LOG_EVAL, "w"),
                        stderr=subprocess.STDOUT, env=dict(os.environ), start_new_session=True)
print("시작. pid", proc.pid, "로그", LOG_EVAL)
# 확인: '시작. pid 숫자'. 로컬 16작업으로 E-GMD가 15분 안팎이었음

In [ ]:
# 7. 결과 확인·묶기 (여러 번 쳐도 됨). 끝나면 zip을 내려받아 로컬 AI/data/results/tom-rule-retune/ 에 풀고
#    로컬에서 scripts/tom_rule_pick.py select·enst 를 돌림 (명령은 AI/README.md)
!grep -c "\[tom-rule\] 저장" $LOG_EVAL
!tail -2 $LOG_EVAL
!wc -l $OUT/stream_egmd_test.csv $OUT/stream_enst_drums.csv $OUT/stream_enst_sep.csv 2>/dev/null
!test -f $OUT/stream_enst_sep.csv && rm -f $WORK/results/tom_rule_retune.zip && cd $WORK/results && $VPY -m zipfile -c tom_rule_retune.zip tom-rule-retune $LOG_FEAT $LOG_EVAL && ls -la tom_rule_retune.zip
# 확인: 저장 줄 3개, 줄 수 107641 / 193 / 193 (머리줄 포함), 수 MB짜리 zip. Traceback이 있으면 마지막 10줄을 적어 둠